# Sentiment Analysis with QLoRA Fine-Tuning — Week 7 Project

## Goal
Apply Week 7 QLoRA fine-tuning techniques to a **new task**: fine-tune Llama 3.1 8B to classify movie review **sentiment** (positive / negative) using the IMDB dataset.

## Pipeline
1. **Data Preparation**: Load IMDB, format as prompt-completion pairs, tokenize, push to Hub  
2. **QLoRA Config**: BitsAndBytesConfig (4-bit), LoraConfig, DataCollatorForCompletionOnlyLM  
3. **Training**: SFTTrainer with SFTConfig, optional W&B logging  
4. **Evaluation**: Load fine-tuned model with PeftModel, run inference, compare accuracy  
5. **Baseline Comparison**: Zero-shot Llama vs TF-IDF + Logistic Regression vs fine-tuned model

## Hardware
Run this notebook on **Google Colab** (T4 or A100 GPU). 4-bit quantized Llama 3.1 8B needs ~5–6 GB VRAM.

---
## 0. Install Dependencies (run in Colab)

In [ ]:
# Uncomment and run in Google Colab
# %pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# %pip install -q --upgrade transformers accelerate bitsandbytes peft trl datasets wandb scikit-learn matplotlib

---
## 1. Data Preparation
Load IMDB movie reviews, format each as a prompt-completion pair.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

set_seed(42)

In [ ]:
# For Colab: read tokens from Colab secrets or env
# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')

# For local: read from .env
from dotenv import load_dotenv
load_dotenv(override=True)

from huggingface_hub import login
hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    login(hf_token, add_to_git_credential=True)

In [ ]:
LITE_MODE = True  # True: 2,000 train examples; False: 25,000 (full IMDB train)
TRAIN_SIZE = 2000 if LITE_MODE else 25000
VAL_SIZE = 500
TEST_SIZE = 1000

In [ ]:
raw = load_dataset("imdb")
print(f"IMDB train: {len(raw['train']):,}, test: {len(raw['test']):,}")

In [ ]:
LABEL_MAP = {0: "negative", 1: "positive"}
QUESTION = "What is the sentiment of this movie review?"
RESPONSE_TEMPLATE = "Sentiment:"
MAX_REVIEW_CHARS = 1500  # truncate long reviews

def format_example(text, label, include_answer=True):
    review = text[:MAX_REVIEW_CHARS].strip()
    prompt = f"{QUESTION}\n\n{review}\n\n{RESPONSE_TEMPLATE}"
    completion = f" {LABEL_MAP[label]}"
    if include_answer:
        return {"prompt": prompt, "completion": completion, "text": prompt + completion}
    return {"prompt": prompt, "completion": completion, "text": prompt}

In [ ]:
train_raw = raw["train"].shuffle(seed=42)
test_raw = raw["test"].shuffle(seed=42)

train_examples = [format_example(train_raw[i]["text"], train_raw[i]["label"]) for i in range(TRAIN_SIZE)]
val_examples = [format_example(train_raw[i]["text"], train_raw[i]["label"]) for i in range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE)]
test_examples = [format_example(test_raw[i]["text"], test_raw[i]["label"], include_answer=False) for i in range(TEST_SIZE)]
test_labels = [LABEL_MAP[test_raw[i]["label"]] for i in range(TEST_SIZE)]

train_ds = Dataset.from_list(train_examples)
val_ds = Dataset.from_list(val_examples)
test_ds = Dataset.from_list(test_examples)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"\nExample train prompt:\n{train_ds[0]['text'][:500]}...")

### Analyze token lengths

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
token_counts = [len(tokenizer.encode(ex["text"])) for ex in train_examples]

plt.figure(figsize=(12, 4))
plt.hist(token_counts, bins=50, color="skyblue", edgecolor="gray")
plt.title(f"Token distribution — Avg: {np.mean(token_counts):.0f}, Max: {max(token_counts)}")
plt.xlabel("Tokens per example")
plt.ylabel("Count")
plt.show()

### (Optional) Push formatted data to HuggingFace Hub

In [ ]:
# HF_USER = "your-username"  # Change to your HuggingFace username
# ds_name = f"{HF_USER}/imdb-sentiment-prompts-lite" if LITE_MODE else f"{HF_USER}/imdb-sentiment-prompts-full"
# DatasetDict({"train": train_ds, "val": val_ds, "test": test_ds}).push_to_hub(ds_name)
# print(f"Pushed to {ds_name}")

---
## 2. Baseline: TF-IDF + Logistic Regression
Quick traditional ML baseline to compare against.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_tfidf = tfidf.fit_transform([train_raw[i]["text"][:MAX_REVIEW_CHARS] for i in range(TRAIN_SIZE)])
y_train_lr = [train_raw[i]["label"] for i in range(TRAIN_SIZE)]

X_test_tfidf = tfidf.transform([test_raw[i]["text"][:MAX_REVIEW_CHARS] for i in range(TEST_SIZE)])
y_test_lr = [test_raw[i]["label"] for i in range(TEST_SIZE)]

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train_lr)
lr_preds = lr_model.predict(X_test_tfidf)

lr_acc = accuracy_score(y_test_lr, lr_preds)
print(f"TF-IDF + Logistic Regression accuracy: {lr_acc:.2%}")
print(classification_report(y_test_lr, lr_preds, target_names=["negative", "positive"]))

---
## 3. Load Base Model with 4-bit Quantization

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

### Zero-shot baseline: ask the base model to predict sentiment before any fine-tuning

In [ ]:
def predict_sentiment(model, tokenizer, prompt, max_new_tokens=5):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=True)
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    if "positive" in generated:
        return "positive"
    elif "negative" in generated:
        return "negative"
    return generated

In [ ]:
ZERO_SHOT_N = 100  # evaluate on a small subset for speed

zs_preds = []
for i in tqdm(range(ZERO_SHOT_N)):
    pred = predict_sentiment(base_model, tokenizer, test_ds[i]["prompt"])
    zs_preds.append(pred)

zs_correct = sum(p == t for p, t in zip(zs_preds, test_labels[:ZERO_SHOT_N]))
print(f"Zero-shot base Llama accuracy ({ZERO_SHOT_N} samples): {zs_correct / ZERO_SHOT_N:.2%}")

---
## 4. QLoRA Configuration and Training

In [ ]:
# DataCollatorForCompletionOnlyLM: only compute loss on tokens after the response template
collator = DataCollatorForCompletionOnlyLM(RESPONSE_TEMPLATE, tokenizer=tokenizer)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in base_model.parameters())
print(f"Before LoRA — Trainable: {trainable:,} / {total:,} ({trainable/total:.2%})")

In [ ]:
# HF_USER = "your-username"  # Change to your HuggingFace username
PROJECT_NAME = "sentiment-qlora"
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

training_config = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",  # change to "wandb" if using Weights & Biases
    fp16=True,
    # hub_model_id=HUB_MODEL_NAME,  # uncomment to push checkpoints to Hub
    # push_to_hub=True,
)

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    args=training_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    peft_config=lora_config,
)

In [ ]:
trainable_after = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total_after = sum(p.numel() for p in trainer.model.parameters())
print(f"After LoRA — Trainable: {trainable_after:,} / {total_after:,} ({trainable_after/total_after:.2%})")

In [ ]:
trainer.train()

In [ ]:
# Save the adapter locally
trainer.save_model(PROJECT_RUN_NAME)
print(f"Adapter saved to {PROJECT_RUN_NAME}/")

# Optional: push to Hub
# trainer.push_to_hub()
# print(f"Pushed to {HUB_MODEL_NAME}")

---
## 5. Evaluation
Load the fine-tuned adapter with PeftModel and evaluate on the test set.

In [ ]:
# If you saved to Hub, load like this:
# fine_tuned = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

# If loading from local checkpoint:
fine_tuned = PeftModel.from_pretrained(base_model, PROJECT_RUN_NAME)
fine_tuned.eval()
print("Fine-tuned model loaded.")

In [ ]:
EVAL_N = min(200, len(test_ds))

ft_preds = []
for i in tqdm(range(EVAL_N)):
    pred = predict_sentiment(fine_tuned, tokenizer, test_ds[i]["prompt"])
    ft_preds.append(pred)

ft_labels_subset = test_labels[:EVAL_N]
ft_acc = accuracy_score(ft_labels_subset, ft_preds)
print(f"\nFine-tuned model accuracy ({EVAL_N} samples): {ft_acc:.2%}")
print(classification_report(ft_labels_subset, ft_preds, target_names=["negative", "positive"], zero_division=0))

---
## 6. Results Comparison

In [ ]:
import plotly.graph_objects as go

results = [
    ("TF-IDF + LogReg", "gray", lr_acc * 100),
    ("Zero-shot Llama 3.1 8B", "slateblue", (zs_correct / ZERO_SHOT_N) * 100),
    ("QLoRA Fine-tuned Llama", "green", ft_acc * 100),
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=list(labels), y=list(values), marker_color=list(colors)))
fig.update_layout(
    title="Sentiment Classification Accuracy (%)",
    yaxis=dict(range=[0, 100], title="Accuracy (%)"),
    xaxis=dict(tickangle=-15),
    width=700,
    height=500,
)
fig.show()

In [ ]:
print("=" * 50)
print("RESULTS SUMMARY")
print("=" * 50)
print(f"  TF-IDF + Logistic Regression : {lr_acc:.2%}")
print(f"  Zero-shot Llama 3.1 8B (4bit): {zs_correct / ZERO_SHOT_N:.2%} ({ZERO_SHOT_N} samples)")
print(f"  QLoRA Fine-tuned Llama       : {ft_acc:.2%} ({EVAL_N} samples)")
print("=" * 50)